# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [39]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [40]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [41]:
step = 1

removed = df.duplicated().sum()

clean = df.drop_duplicates().copy()

log(step, "dropped duplicates", int(removed))


[1] dropped duplicates (15 row(s))


### TODO 2 — clean `price` -> float

In [42]:
step = 2

before = clean['price'].astype(str)
needs_clean = before.str.contains(r'\$') | before.str.contains(' ')

rows_affected = int(needs_clean.sum())

clean['price'] = (
    clean['price']
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.strip()
        .astype(float)
)

assert clean['price'].dtype == float

log(step, "cleaned price to float", rows_affected)
clean['price'].astype(str).unique()


[2] cleaned price to float (124 row(s))


array(['12.0', '7.5', '6.0', '24.0'], dtype=object)

### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [43]:
step = 3

clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()
negative = (clean['qty'] < 0).sum()

clean = clean.dropna(subset=['qty'])
clean = clean[clean['qty'] >= 0]

rows_affected = int(missing + negative)

log(step, "cleaned qty column", rows_affected)


[3] cleaned qty column (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [44]:
step = 4

before = sorted(clean['item'].astype(str).unique())

clean['item'] = (
    clean['item']
        .astype(str)
        .str.lower()
        .str.strip()
)

ITEM_MAP = {
    'cheese burger': 'cheeseburger',
    'cheeseburger': 'cheeseburger',
    'foam finger': 'foam finger',
    'rain poncho': 'rain poncho'
}

before_distinct = len(before)

clean['item'] = clean['item'].replace(ITEM_MAP)

after_distinct = clean['item'].nunique()

rows_affected = before_distinct - after_distinct

log(step, "canonicalized item names", rows_affected)


[4] canonicalized item names (3 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [45]:
step = 5

before = sorted(clean['category'].astype(str).unique())

clean['category'] = (
    clean['category']
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace('-', ' ', regex=False)
)

CATEGORY_MAP = {
    'food': 'food',
    'merch': 'merch',
    'apparel': 'merch',
    'rain gear': 'rain gear',
    'raingear': 'rain gear'
}

before_distinct = len(before)

clean['category'] = clean['category'].replace(CATEGORY_MAP)

after_distinct = clean['category'].nunique()

rows_affected = before_distinct - after_distinct

log(step, "normalized category", rows_affected)


[5] normalized category (3 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [46]:
step = 6

assert clean.duplicated().sum() == 0
assert clean['qty'].min() >= 1
assert clean['price'].dtype == float

assert set(clean['item'].unique()) == {'cheeseburger', 'foam finger', 'rain poncho'}
assert set(clean['category'].unique()) == {'food', 'merch', 'rain gear'}

print('clean:', clean.shape)


clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [47]:
step = 7

clean['revenue'] = clean['qty'] * clean['price']

rev_by_cat = (
    clean.groupby('category')['revenue']
         .sum()
         .round(2)
         .sort_values(ascending=False)
)

total_rev = round(clean['revenue'].sum(), 2)

print(rev_by_cat)
print("total:", total_rev)


category
food         1656.0
merch        1572.0
rain gear    1512.0
Name: revenue, dtype: float64
total: 4740.0


**What I would tell the vendor:** Food edges out the others, so the venor should stock more food because it reliably generates the highest revenue and sells fastest.

### TODO 8 — read back your log

In [48]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,1,dropped duplicates,15
1,2,cleaned price to float,124
2,3,cleaned qty column,25
3,4,canonicalized item names,3
4,5,normalized category,3


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

A) The cleaning step that changed my revenue total the most was Step 2, where the price column was cleaned and converted to float. Before this step, the total revenue was 3504.0; after cleaning the price values, the total increased to 4740.0. No other step produced a change of similar magnitude.

B)A reasonable person could have chosen to keep Apparel separate from Merch instead of collapsing both into a single merch category. Keeping Apparel distinct would have lowered the reported merch revenue and split the middle category into two smaller ones. I chose to merge them because stadium vendors typically treat apparel and merch as one operational and reporting group, so combining them reflects the business reality rather than the raw string variation in the data.